---
# Market Risk Analysis
## VaR, Expected Shortfall & Stress Testing

A multi-asset portfolio risk framework covering instrument 
pricing, historical simulation, stress testing, and 
regulatory risk measures.

**Stack:** Python · NumPy · Pandas · Matplotlib · Seaborn  
**Data:** 502 daily observations, Jan 2021 – Dec 2022  
**Author:** Mohammad Valizadehmoghadam
---


---
## Table of Contents
1. [Data & Setup](#setup)
2. [Instrument Pricing](#pricing)
3. [Risk Sensitivities](#sensitivities)
4. [Historical Shock Analysis](#shocks)
5. [Value-at-Risk & Expected Shortfall](#var)
6. [VaR Backtesting](#backtesting)
7. [Stress Testing](#stress)
8. [Risk Limits](#limits)
9. [DV01 by Tenor](#dv01)
10. [Risk Factor Correlations](#correlation)
11. [FRTB Liquidity Horizons](#frtb)
---


<a id='setup'></a>

## 1. Data & Setup
Loads market data, defines portfolio constants, and sets 
visualization style.


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import os
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker


In [ ]:
# --- Visualization style ---
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#f8f8f8',
    'axes.edgecolor':   '#cccccc',
    'axes.grid':        True,
    'grid.color':       'white',
    'grid.linewidth':   0.8,
    'font.family':      'sans-serif',
    'font.size':        11,
    'axes.titlesize':   12,
    'axes.titleweight': 'bold',
    'axes.labelsize':   10,
    'xtick.labelsize':  9,
    'ytick.labelsize':  9,
})
PALETTE = ['#2C5F8A', '#E07B39', '#4A9B6F', '#C94040', '#7B5EA7', '#5A9BAA']



In [ ]:
df = pd.read_excel("data/market data time series.xlsx")
df.head()


In [ ]:
#setting the base variables
COUPONS = {"IG": 5e-2,
          "HY": 7e-2,
          "UST": 3.875e-2}  # decimal fraction
UST_YIELD = {"10y": 3.56e-2,
             "5y": 3.7e-2,
             "2y": 4.25e-2}  # decimal fraction
CREDIT_SPREAD = {"IG": 112e-4,
                 "HY": 295e-4,
                 "UST": 0}  # decimal fraction
NUM_POSITIONS = {"SPY": 30,
                 "IG": 1e4,
                 "HY": 1e4,
                 "UST": -2e4}
SPY_PRESENT_PRICE = 380.82  # price per unit SPY, USD
COUPON_PAY_FREQ = 2  # twice per year, i.e., semi-annual, for all instrument in portfolio
BOND_NOTIONAL = 100  # notional in USD


In [ ]:
# Stress scenario shock inputs (decimal)
CS_SHOCK = {"AAA": 238.9e-4,
            "AA": 266.7e-4,
            "A": 337.6e-4,
            "BBB": 443e-4,
            "BB": 1012e-4,
            "B": 1372.1e-4}  # decimal fraction
UST_SHOCK = {"10y": 49.4e-4,
             "5y": 42.3e-4,
             "2y": 29.9e-4}  # decimal fraction
SPY_SHOCK = -38.3e-2  # decimal fraction


<a id='pricing'></a>

## 2. Instrument Pricing
Prices each instrument using continuous compounding 
discounting.


In [ ]:
def compute_bond_price(instrument, year, eps=0):
    """
    This function computes the present value per position for a given bond, with a given time to mature.
    :param instrument: bond name
    :param year: time to mature
    :param eps: shock, default is 0. This will be used for sensitivity analysis later.
    :return present value per position
    """

    # Retrieve coupon rate and build discount rate
    # Discount rate = risk-free yield + credit spread (both in decimal)
    # eps shifts the discount rate for sensitivity calculations
    
    coupon_rate = COUPONS[instrument]  # already in decimal (e.g. 0.05)
    
    if instrument == 'IG':
        r = UST_YIELD['10y'] + CREDIT_SPREAD['IG'] + eps
    elif instrument == 'HY':
        r = UST_YIELD['2y'] + CREDIT_SPREAD['HY'] + eps
    elif instrument == 'UST':
        r = UST_YIELD['5y'] + CREDIT_SPREAD['UST'] + eps
    
    # Semi-annual payments: t = 0.5, 1.0, 1.5, ..., maturity
    coupon_payment = (coupon_rate * BOND_NOTIONAL) / COUPON_PAY_FREQ
    num_periods = int(year * COUPON_PAY_FREQ)  # total number of coupon payments
    times = np.array([i / COUPON_PAY_FREQ for i in range(1, num_periods + 1)])
    
    # Discount each cash flow
    # Coupon PV: sum of C * e^(-r * t) for each period
    # Principal PV: F * e^(-r * T) at maturity only
    pv_coupons = np.sum(coupon_payment * np.exp(-r * times))
    pv_principal = BOND_NOTIONAL * np.exp(-r * year)
    
    unit_present_price = pv_coupons + pv_principal
    
    return unit_present_price


In [ ]:
present_price_per_position_SPY = SPY_PRESENT_PRICE #no need for discounting
#present value per position for each instrument using continuous compounding
present_price_per_position_IG = compute_bond_price('IG',  year=10)
present_price_per_position_HY = compute_bond_price('HY',  year=2)
present_price_per_position_UST = compute_bond_price('UST',  year=5)


In [ ]:
print("Present value per position:")
print("SPY ETF: ${}".format("%.2f" % present_price_per_position_SPY))
print("IG Corporate bond (10 years): ${}".format("%.2f" % present_price_per_position_IG))
print("HY Corporate bond (2 years): ${}".format("%.2f" % present_price_per_position_HY))
print("US Treasury (5 years): ${}".format("%.2f" % present_price_per_position_UST))


In [ ]:
# Export results
present_price_per_position = {
    "SPY": present_price_per_position_SPY,
    "IG": present_price_per_position_IG,
    "HY": present_price_per_position_HY,
    "UST": present_price_per_position_UST
}
present_price_per_position_df = pd.DataFrame.from_dict(present_price_per_position, orient='index', columns=['Present Price'])
os.makedirs('output', exist_ok=True)
present_price_per_position_df.to_csv('output/present_price_per_position.csv')


<a id='sensitivities'></a>

## 3. Risk Sensitivities
Computes first-order sensitivities: Equity Delta, PV01, 
and CS01.


In [ ]:
def func_shock(instrument, years, eps=0):
  """
  This function computes the total shock for all positions for a given bond, with a given time to mature.
  :param instrument: bond name
  :param years: time to mature
  :param eps: shock 
  :return total shock for all positions
  """
  # Price change per position under the shock
  price_change = compute_bond_price(instrument, years, eps=eps) - \
                 compute_bond_price(instrument, years, eps=0)
    
  # Multiply by number of positions
  total_shock = price_change * NUM_POSITIONS[instrument]
    
  return total_shock


In [ ]:
# Equity Delta
# Delta = positions × current price × 1%
equity_delta = NUM_POSITIONS['SPY'] * SPY_PRESENT_PRICE * 0.01

# PV01: P&L impact of +1 bps parallel shift in risk-free yields 
# 1 bps = 0.0001 in decimal; applied to ALL bonds (rates affect everyone)
eps_1bps = 1e-4
PV01 = (func_shock('IG',  years=10, eps=eps_1bps) +
        func_shock('HY',  years=2,  eps=eps_1bps) +
        func_shock('UST', years=5,  eps=eps_1bps))

# CS01: P&L impact of +1 bps widening in credit spreads 
# Only affects credit instruments (IG and HY), not the UST
CS01 = (func_shock('IG', years=10, eps=eps_1bps) +
        func_shock('HY', years=2,  eps=eps_1bps))


In [ ]:
print("First-order sensitivity:")
print("Equity Delta: ${}".format("%.2f" % equity_delta))
print("PV01: ${}".format("%.2f" % PV01))
print("CS01: ${}".format("%.2f" % CS01))


In [ ]:
# Export results
sensitivity = {
    "Equity Delta": equity_delta,
    "PV01": PV01,
    "CS01": CS01
}
sensitivity_df = pd.DataFrame.from_dict(sensitivity, orient='index', columns=['Sensitivity'])
sensitivity_df.to_csv('output/sensitivity.csv')


<a id='shocks'></a>

## 4. Historical Shock Analysis
Derives daily risk factor shocks from the 2021–2022 
time series and plots their distributions.


In [ ]:
# Sorting by date ascending so that the differences go forward in time
df_sorted = df.sort_values('Date').reset_index(drop=True)

daily_shock_SPY =  df_sorted['SPY ETF ($)'].pct_change().dropna().values# relative shock (percent)
daily_shock_UST_IG_10 = df_sorted['Risk-free Yield @ 10y (%)'].diff().dropna().values * 100 # absolute shock (bps)
daily_shock_UST_5 =  df_sorted['Risk-free Yield @ 5y (%)'].diff().dropna().values  * 100 # absolute shock (bps)
daily_shock_UST_HY_2 = df_sorted['Risk-free Yield @ 2y (%)'].diff().dropna().values  * 100  # absolute shock (bps)
daily_shock_CS_IG =   df_sorted['Credit Spread IG (bps)'].diff().dropna().values # absolute shock (bps)
daily_shock_CS_HY =   df_sorted['Credit Spread HY (bps)'].diff().dropna().values # absolute shock (bps)


print(f"Number of shock scenarios: {len(daily_shock_SPY)}")
print(f"\nSPY shock range:    {daily_shock_SPY.min()*100:.2f}% to {daily_shock_SPY.max()*100:.2f}%")
print(f"UST 10y shock range: {daily_shock_UST_IG_10.min():.2f} to {daily_shock_UST_IG_10.max():.2f} bps")
print(f"CS IG shock range:   {daily_shock_CS_IG.min():.2f} to {daily_shock_CS_IG.max():.2f} bps")
print(f"CS HY shock range:   {daily_shock_CS_HY.min():.2f} to {daily_shock_CS_HY.max():.2f} bps")


In [ ]:
# Export results
daily_shock = {
    "SPY": daily_shock_SPY,
    "UST_10": daily_shock_UST_IG_10,
    "UST_5": daily_shock_UST_5,
    "UST_2": daily_shock_UST_HY_2,
    "CS_IG": daily_shock_CS_IG,
    "CS_HY": daily_shock_CS_HY
}
daily_shock_df = pd.DataFrame.from_dict(daily_shock)
daily_shock_df.to_csv('output/daily_shock.csv', index=False)


In [ ]:
fig, axes = plt.subplots(6, 1, figsize=(10, 16))
fig.suptitle('Daily Risk Factor Shock Distributions\n2021-01-04 to 2022-12-29 (501 scenarios)',
             fontsize=14, fontweight='bold', y=1.01)
plt.subplots_adjust(hspace=0.6)

shock_data = [
    (daily_shock_SPY * 100, 'SPY Equity Return (%)', PALETTE[0]),
    (daily_shock_UST_IG_10, 'UST 10yr Yield Change (bps)', PALETTE[1]),
    (daily_shock_UST_HY_2,  'UST 2yr Yield Change (bps)',  PALETTE[2]),
    (daily_shock_UST_5,     'UST 5yr Yield Change (bps)',  PALETTE[3]),
    (daily_shock_CS_IG,     'IG Credit Spread Change (bps)', PALETTE[4]),
    (daily_shock_CS_HY,     'HY Credit Spread Change (bps)', PALETTE[5]),
]

for ax, (data, label, color) in zip(axes, shock_data):
    ax.hist(data, bins=40, color=color, edgecolor='white', linewidth=0.4, alpha=0.85)
    ax.axvline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
    ax.set_xlabel(label)
    ax.set_ylabel('Frequency')
    mean_val = data.mean()
    std_val  = data.std()
    ax.set_title(f'{label}  |  mean={mean_val:.3f}  std={std_val:.3f}')

plt.savefig('output/daily_shock_histogram.png', bbox_inches='tight', dpi=150)
plt.show()



<a id='var'></a>

## 5. Value-at-Risk & Expected Shortfall
Computes 1-day 99% VaR (6th worst scenario) and 97.5% 
ES (average of 13 worst) using the Taylor approach.


In [ ]:
# Date index aligned with shock scenarios 
dates = df_sorted['Date'].iloc[1:].reset_index(drop=True)
# --- Compute per-1bps sensitivities separately ---
# Rate sensitivity (PV01 component per instrument)
pv01_IG  = func_shock('IG',  years=10, eps=eps_1bps)  # $ per 1bps rate move
pv01_HY  = func_shock('HY',  years=2,  eps=eps_1bps)  # $ per 1bps rate move
pv01_UST = func_shock('UST', years=5,  eps=eps_1bps)  # $ per 1bps rate move

# Credit spread sensitivity (CS01 component per instrument)
cs01_IG  = func_shock('IG',  years=10, eps=eps_1bps)  # $ per 1bps spread move
cs01_HY  = func_shock('HY',  years=2,  eps=eps_1bps)  # $ per 1bps spread move
# UST has no credit spread — cs01_UST = 0

# --- SPY P&L ---
# equity_delta is $ sensitivity to 1% move, scale by actual shock
pnl_SPY = equity_delta * (daily_shock_SPY / 0.01)

# --- IG Bond P&L: rate component + credit spread component ---
pnl_IG = (pv01_IG * daily_shock_UST_IG_10 +
          cs01_IG * daily_shock_CS_IG)

# --- HY Bond P&L: rate component + credit spread component ---
pnl_HY = (pv01_HY * daily_shock_UST_HY_2 +
          cs01_HY * daily_shock_CS_HY)

# --- UST P&L: rate only, no credit spread ---
pnl_UST = pv01_UST * daily_shock_UST_5

# --- Portfolio P&L ---
pnl_portfolio = pnl_SPY + pnl_IG + pnl_HY + pnl_UST



In [ ]:
### Compute 99th percentile for P&L for each instrument
var99_SPY       = pnl_SPY[np.argsort(pnl_SPY)[5]]
var99_IG        = pnl_IG[np.argsort(pnl_IG)[5]]
var99_HY        = pnl_HY[np.argsort(pnl_HY)[5]]
var99_UST       = pnl_UST[np.argsort(pnl_UST)[5]]
var99_portfolio = pnl_portfolio[np.argsort(pnl_portfolio)[5]]

date_SPY       = dates.iloc[np.argsort(pnl_SPY)[5]].strftime('%Y-%m-%d')
date_IG        = dates.iloc[np.argsort(pnl_IG)[5]].strftime('%Y-%m-%d')
date_HY        = dates.iloc[np.argsort(pnl_HY)[5]].strftime('%Y-%m-%d')
date_UST       = dates.iloc[np.argsort(pnl_UST)[5]].strftime('%Y-%m-%d')
date_portfolio = dates.iloc[np.argsort(pnl_portfolio)[5]].strftime('%Y-%m-%d')

var99 = {
    "SPY":       [var99_SPY, date_SPY],
    "IG":        [var99_IG, date_IG],
    "HY":        [var99_HY, date_HY],
    "UST":       [var99_UST, date_UST],
    "Portfolio": [var99_portfolio, date_portfolio]
}

var99_df = round(pd.DataFrame.from_dict(var99, orient='index', columns=['99th Percentile', 'Date']), 2)
display(var99_df)


In [ ]:
# Export results
os.makedirs('output', exist_ok=True)
var99_df.to_csv('output/var99.csv')


In [ ]:
### Compute Expected Shortfall at 97.5 percentile using np.sort or np.percentile

sorted_pnl = np.sort(pnl_portfolio)  # ascending, worst first

ES_975 = np.mean(sorted_pnl[:13])

print("Expected Shortfall at 97.5th percentile: ${:,.0f}".format(ES_975))
print("(Average of the 13 worst daily P&L scenarios)")
print(f"\nFor reference, 99th percentile VaR was: ${var99_portfolio:,.0f}")
print(f"ES/VaR ratio: {ES_975/var99_portfolio:.2f}x")


In [ ]:
# P&L Distribution with VaR and ES marked
fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(pnl_portfolio, bins=50, color=PALETTE[0], edgecolor='white',
        linewidth=0.4, alpha=0.85, label='Daily P&L')
ax.axvline(var99_portfolio, color=PALETTE[3], linewidth=2,
           linestyle='--', label=f'99% VaR: ${var99_portfolio:,.0f}')
ax.axvline(ES_975, color=PALETTE[1], linewidth=2,
           linestyle='--', label=f'97.5% ES: ${ES_975:,.0f}')
ax.fill_betweenx([0, ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 100],
                  pnl_portfolio.min(), var99_portfolio,
                  color=PALETTE[3], alpha=0.08, label='Tail region')
ax.set_xlabel('Daily Portfolio P&L (USD)')
ax.set_ylabel('Frequency')
ax.set_title('Portfolio P&L Distribution — Historical Simulation (501 Scenarios)\nwith 99% VaR and 97.5% ES')
ax.legend(framealpha=0.9)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.savefig('output/pnl_distribution.png', bbox_inches='tight', dpi=150)
plt.show()



In [ ]:
# Export results
expected_shortfall = {
    "Portfolio": ES_975
}
expected_shortfall_df = pd.DataFrame.from_dict(expected_shortfall, orient='index', columns=['Expected Shortfall'])
expected_shortfall_df.to_csv('output/expected_shortfall.csv')


<a id='backtesting'></a>

## 6. VaR Backtesting
Validates the VaR model by comparing the 99% threshold 
against realized daily P&L across all 501 scenarios.


In [ ]:
# VaR Backtesting — compare rolling VaR threshold against realized P&L
var_threshold = var99_portfolio  # our computed 1-day 99% VaR (negative number)

# Identify breach days
breaches = pnl_portfolio < var_threshold
n_breaches = breaches.sum()
breach_dates = dates[breaches].values
breach_pnl   = pnl_portfolio[breaches]

print(f"VaR Threshold (99%, 1-day): ${var_threshold:,.0f}")
print(f"Total scenarios: 501")
print(f"Number of VaR breaches: {n_breaches}")
print(f"Breach rate: {n_breaches/501*100:.2f}%")
print(f"Expected at 99% confidence: ~5 breaches")
print()
if n_breaches <= 4:
    zone = "GREEN — model is conservative"
elif n_breaches <= 9:
    zone = "GREEN — model is performing within expectations"
elif n_breaches <= 12:
    zone = "YELLOW — elevated breaches, model review recommended"
else:
    zone = "RED — excessive breaches, capital add-on required"
print(f"Basel III Traffic Light: {zone}")

# Backtesting chart
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(dates, pnl_portfolio, color=PALETTE[0], linewidth=0.8,
        alpha=0.8, label='Daily Portfolio P&L')
ax.axhline(var_threshold, color=PALETTE[3], linewidth=1.5,
           linestyle='--', label=f'99% VaR threshold: ${var_threshold:,.0f}')
ax.scatter(dates[breaches], breach_pnl, color=PALETTE[3],
           zorder=5, s=60, label=f'VaR breaches ({n_breaches})', marker='x')
ax.fill_between(dates, pnl_portfolio, var_threshold,
                where=(pnl_portfolio < var_threshold),
                color=PALETTE[3], alpha=0.3)
ax.set_xlabel('Date')
ax.set_ylabel('Daily P&L (USD)')
ax.set_title('VaR Backtesting — 501 Historical Scenarios (2021–2022)\n99% Confidence Level')
ax.legend(framealpha=0.9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.savefig('output/var_backtesting.png', bbox_inches='tight', dpi=150)
plt.show()



<a id='stress'></a>

## 7. Stress Testing
Applies a severely adverse macro scenario using full 
revaluation and Taylor approximation, then compares results.


In [ ]:
# SPY: full revaluation
SPY_shocked_price = SPY_PRESENT_PRICE * (1 + SPY_SHOCK)
SPY_shock_full_reval = (SPY_shocked_price - SPY_PRESENT_PRICE) * NUM_POSITIONS['SPY']

# Bonds: func_shock with combined rate + credit spread shock
IG_shock_full_reval  = func_shock('IG',  years=10, eps=UST_SHOCK['10y'] + CS_SHOCK['A'])
HY_shock_full_reval  = func_shock('HY',  years=2,  eps=UST_SHOCK['2y']  + CS_SHOCK['BB'])
UST_shock_full_reval = func_shock('UST', years=5,  eps=UST_SHOCK['5y'])

# Portfolio total
portfolio_shock_full_reval = (SPY_shock_full_reval + IG_shock_full_reval + 
                               HY_shock_full_reval + UST_shock_full_reval)


In [ ]:
full_reval = {
    "SPY": SPY_shock_full_reval,
    "IG": IG_shock_full_reval,
    "HY": HY_shock_full_reval,
    "UST": UST_shock_full_reval,
    "Portfolio": portfolio_shock_full_reval
}
full_reval_df = round(pd.DataFrame.from_dict(full_reval, orient='index', columns=['Stress Testing P&L (Full Reval)'])
    , 2)

display(full_reval_df)


In [ ]:
# Risk contribution bar chart
fig, ax = plt.subplots(figsize=(9, 5))
instruments  = ['SPY', 'IG Bond', 'HY Bond', 'UST (Short)']
contributions = [SPY_shock_full_reval, IG_shock_full_reval,
                 HY_shock_full_reval,  UST_shock_full_reval]
bar_colors = [PALETTE[3] if v < 0 else PALETTE[2] for v in contributions]
bars = ax.bar(instruments, contributions, color=bar_colors,
              edgecolor='white', linewidth=0.5, width=0.5)
ax.axhline(0, color='black', linewidth=0.8)
for bar, val in zip(bars, contributions):
    ypos = val - 8000 if val < 0 else val + 3000
    ax.text(bar.get_x() + bar.get_width()/2, ypos,
            f'${val:,.0f}', ha='center', va='top' if val < 0 else 'bottom',
            fontsize=10, fontweight='bold', color='white' if val < 0 else 'black')
ax.set_ylabel('Stress P&L (USD)')
ax.set_title('Stress Test P&L by Instrument — Full Revaluation\nSeverely Adverse Scenario')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.savefig('output/stress_contribution.png', bbox_inches='tight', dpi=150)
plt.show()



In [ ]:
# Export results
full_reval_df.to_csv('output/stress_testing_full_reval.csv')


In [ ]:
### compute profit and loss (P&L), Taylor approach

SPY_shock_taylor = equity_delta * (SPY_SHOCK / 0.01)

# Bonds: sensitivity × shock in bps
# Rate shock component (PV01 × rate shock)
IG_shock_taylor  = (func_shock('IG',  years=10, eps=eps_1bps) * UST_SHOCK['10y']  * 1e4 +
                    func_shock('IG',  years=10, eps=eps_1bps) * CS_SHOCK['A']     * 1e4)

HY_shock_taylor  = (func_shock('HY',  years=2,  eps=eps_1bps) * UST_SHOCK['2y']   * 1e4 +
                    func_shock('HY',  years=2,  eps=eps_1bps) * CS_SHOCK['BB']    * 1e4)

UST_shock_taylor = func_shock('UST', years=5,  eps=eps_1bps) * UST_SHOCK['5y']   * 1e4

# Portfolio total
portfolio_shock_taylor = (SPY_shock_taylor + IG_shock_taylor + 
                          HY_shock_taylor  + UST_shock_taylor)


In [ ]:
taylor = {
    "SPY": SPY_shock_taylor,
    "IG": IG_shock_taylor,
    "HY": HY_shock_taylor,
    "UST": UST_shock_taylor,
    "Portfolio": portfolio_shock_taylor
}
taylor_df = round(pd.DataFrame.from_dict(taylor, orient='index', columns=['Stress Testing P&L (Taylor)']), 2)
display(taylor_df)


In [ ]:
# Export results
taylor_df.to_csv('output/stress_testing_taylor.csv')


<a id='limits'></a>

## 8. Risk Limits
Proposed risk limit framework derived from computed 
VaR, ES, and stress testing results.


In [ ]:
limits_data = {
    'Limit Type': [
        'VaR Limit (1-day, 99%)',
        'Stress Loss Limit',
        'Equity Delta Limit',
        'PV01 Limit (per bps)',
        'CS01 Limit (per bps)'
    ],
    'Limit Value': [
        '$15,000',
        '$500,000',
        '±$500',
        '±$200',
        '±$1,500'
    ],
    'Current Exposure': [
        f'${abs(var99_portfolio):,.0f}',
        f'${abs(portfolio_shock_full_reval):,.0f}',
        f'${equity_delta:,.0f}',
        f'${abs(PV01):,.0f}',
        f'${abs(CS01):,.0f}'
    ],
    'Headroom': [
        f'${15000 - abs(var99_portfolio):,.0f}',
        f'${500000 - abs(portfolio_shock_full_reval):,.0f}',
        f'${500 - equity_delta:,.0f}',
        f'${200 - abs(PV01):,.0f}',
        f'${1500 - abs(CS01):,.0f}'
    ]
}

limits_df = pd.DataFrame(limits_data)
display(limits_df)


See report for full analysis.


<a id='dv01'></a>

## 9. DV01 by Tenor
Breaks down rate sensitivity by yield curve point: 
2yr, 5yr, and 10yr.


In [ ]:
# DV01 at each tenor: P&L impact of +1bps move at that curve point only
# Each instrument is sensitive to one specific tenor

dv01_2y  = pv01_HY   # HY bond discounted at 2yr rate
dv01_5y  = pv01_UST  # UST discounted at 5yr rate  
dv01_10y = pv01_IG   # IG bond discounted at 10yr rate

dv01_df = pd.DataFrame({
    'Tenor':       ['2yr', '5yr', '10yr'],
    'Instrument':  ['HY Corporate Bond', 'US Treasury (Short)', 'IG Corporate Bond'],
    'DV01 ($/bps)': [round(dv01_2y, 2), round(dv01_5y, 2), round(dv01_10y, 2)],
    'Abs DV01':    [abs(round(dv01_2y, 2)), abs(round(dv01_5y, 2)), abs(round(dv01_10y, 2))]
})
display(dv01_df[['Tenor', 'Instrument', 'DV01 ($/bps)']])

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4))
colors = [PALETTE[2] if v >= 0 else PALETTE[3] for v in 
          [dv01_2y, dv01_5y, dv01_10y]]
ax.bar(dv01_df['Tenor'], dv01_df['DV01 ($/bps)'], 
       color=colors, width=0.4, edgecolor='white')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('Yield Curve Tenor')
ax.set_ylabel('DV01 (USD per bps)')
ax.set_title('DV01 by Tenor Bucket\nRate Sensitivity Across the Yield Curve')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
for i, (tenor, val) in enumerate(zip(dv01_df['Tenor'], 
                                      dv01_df['DV01 ($/bps)'])):
    ax.text(i, val + (200 if val >= 0 else -200), f'${val:,.0f}',
            ha='center', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig('output/dv01_by_tenor.png', bbox_inches='tight', dpi=150)
plt.show()



<a id='correlation'></a>

## 10. Risk Factor Correlations
Computes and visualizes the correlation matrix across 
all six daily risk factor shocks.


In [ ]:
# Build risk factor shock dataframe
rf_df = pd.DataFrame({
    'SPY Return (%)':     daily_shock_SPY * 100,
    'UST 2yr (bps)':      daily_shock_UST_HY_2,
    'UST 5yr (bps)':      daily_shock_UST_5,
    'UST 10yr (bps)':     daily_shock_UST_IG_10,
    'CS IG (bps)':        daily_shock_CS_IG,
    'CS HY (bps)':        daily_shock_CS_HY,
})

corr = rf_df.corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.zeros_like(corr, dtype=bool)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, vmin=-1, vmax=1,
            linewidths=0.5, linecolor='white',
            ax=ax, annot_kws={'size': 10})
ax.set_title('Risk Factor Shock Correlation Matrix\n2021–2022 Historical Window',
             fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('output/correlation_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()



<a id='frtb'></a>

## 11. FRTB Liquidity Horizons
Scales 1-day VaR to FRTB regulatory holding periods 
using the square root of time rule.


In [ ]:
# Decompose 1-day VaR into risk factor contributions
var_SPY = var99_SPY
var_IG  = var99_IG
var_HY  = var99_HY
var_UST = var99_UST

# FRTB liquidity horizons
lh_equity = 10
lh_ig_cs  = 40
lh_hy_cs  = 60
lh_rates  = 20

# Scale each component by its liquidity horizon
scaled_SPY = var_SPY * np.sqrt(lh_equity)
scaled_IG  = var_IG  * np.sqrt(lh_ig_cs)
scaled_HY  = var_HY  * np.sqrt(lh_hy_cs)
scaled_UST = var_UST * np.sqrt(lh_rates)

lh_df = pd.DataFrame({
    'Instrument':          ['SPY', 'IG Bond', 'HY Bond', 'UST (Short)'],
    'Risk Class':          ['Equity', 'IG Credit', 'HY Credit', 'Rates'],
    'Liquidity Horizon':   [lh_equity, lh_ig_cs, lh_hy_cs, lh_rates],
    '1-day VaR':           [round(var_SPY, 0), round(var_IG, 0),
                            round(var_HY, 0), round(var_UST, 0)],
    'Scaled VaR':          [round(scaled_SPY, 0), round(scaled_IG, 0),
                            round(scaled_HY, 0), round(scaled_UST, 0)],
})
display(lh_df)

